In [1]:
import sqlite3                                 
import pandas as pd                             
 
con = sqlite3.connect('../data/raw/airtel.db')  
 
def q(sql):                                     

    return pd.read_sql(sql, con)

In [2]:
connection_counts = q("""
    SELECT connection_type, COUNT(*) AS subscribers
    FROM customer_features
    GROUP BY connection_type
""")

connection_counts["share"] = connection_counts["subscribers"] / 22_000
connection_counts["share_pct"] = connection_counts["share"] * 100
connection_counts

,connection_type,subscribers,share,share_pct
0,Postpaid,4851,0.2205,22.05
1,Prepaid,17149,0.7795,77.95


In [3]:
top_plans = q("""
    SELECT plan_name, plan_type, price_inr, COUNT(*) AS subscribers
    FROM customer_features
    GROUP BY plan_name, plan_type, price_inr
    ORDER BY subscribers DESC
    LIMIT 5
""")

top_plans

,plan_name,plan_type,price_inr,subscribers
0,Prepaid Annual 1799,Prepaid,1799,2901
1,Prepaid Quarterly 719,Prepaid,719,2561
2,Prepaid Max 479,Prepaid,479,2392
3,Prepaid Entertainment 399,Prepaid,399,2142
4,Prepaid Data 349,Prepaid,349,1884


In [4]:
top_circles = q("""
    SELECT circle_name, region, COUNT(*) AS subscribers
    FROM customer_features
    GROUP BY circle_name, region
    ORDER BY subscribers DESC
    LIMIT 5
""")

top_circles

,circle_name,region,subscribers
0,Maharashtra,West,1428
1,Andhra Pradesh,South,1412
2,Mumbai,West,1395
3,Karnataka,South,1361
4,North East,East,1318


In [5]:
q("""
SELECT
    age_band,
    COUNT(*) AS subscriber_count,
    ROUND(COUNT(*) * 100.0 / 22000, 2) AS share_percentage
FROM customer_features
GROUP BY age_band
ORDER BY subscriber_count DESC;
""")

,age_band,subscriber_count,share_percentage
0,25-34,7467,33.94
1,18-24,5268,23.95
2,35-44,4824,21.93
3,45-54,2895,13.16
4,55+,1546,7.03


In [6]:
q("""
SELECT
    acquisition_channel,
    COUNT(*) AS subscriber_count,
    ROUND(COUNT(*) * 100.0 / 22000, 2) AS share_percentage
FROM customer_features
GROUP BY acquisition_channel
ORDER BY subscriber_count DESC;
""")

,acquisition_channel,subscriber_count,share_percentage
0,Retailer,9287,42.21
1,Airtel App,6513,29.60
2,Website,3081,14.00
3,Call Center,1770,8.05
4,Doorstep,1349,6.13


In [7]:
q("""
SELECT
    device_type,
    os,
    COUNT(*) AS subscriber_count
FROM customer_features
GROUP BY device_type,os
ORDER BY device_type,os""")

,device_type,os,subscriber_count
0,Feature Phone,Other,3029
1,Smartphone,Android,15613
2,Smartphone,iOS,3358


In [8]:
q("""
SELECT
    tenure_years,
    COUNT(*) AS subscriber_count,
    ROUND(COUNT(*) * 100.0 / 22000, 2) AS share_percentage
FROM customer_features
GROUP BY tenure_years
ORDER BY subscriber_count DESC;
""")

,tenure_years,subscriber_count,share_percentage
0,2,7055,32.07
1,1,6961,31.64
2,0,4610,20.95
3,3,3374,15.34


In [9]:
q("""
SELECT
    round(avg(stated_arpu), 2) AS average_arpu
FROM customer_features;
""")

,average_arpu
0,421.65


In [10]:
q("""SELECT
    AVG(actual_arpu_6m) AS average_actual_arpu,
    SUM(actual_arpu_6m) AS total_monthly_revenue
FROM customer_features;""")

,average_actual_arpu,total_monthly_revenue
0,344.464097,7578210.14


In [11]:
q("""select
ROUND(AVG(stated_arpu),2)as avg_st_arpu, 
ROUND(AVG(actual_arpu_6m),2)as act_arpu_6pm,
ROUND(AVG(stated_arpu - actual_arpu_6m),2)as gap,
ROUND(
        AVG(stated_arpu - actual_arpu_6m)
        * 100.0 / AVG(actual_arpu_6m), 2
    ) AS gap_percentage
FROM customer_features;
""")

,avg_st_arpu,act_arpu_6pm,gap,gap_percentage
0,421.65,344.46,77.18,22.41


In [12]:
q("""SELECT
    plan_name,
    COUNT(*) AS total_subscribers,
    SUM(actual_arpu_6m) AS total_monthly_revenue,
    AVG(actual_arpu_6m) AS avg_revenue_per_subscriber
FROM customer_features
GROUP BY plan_name
ORDER BY total_monthly_revenue DESC
LIMIT 5;
""")

,plan_name,total_subscribers,total_monthly_revenue,avg_revenue_per_subscriber
0,Postpaid Infinity 1499,1431,2171756.20,1517.649336
1,Postpaid Family 999,1239,1249245.94,1008.269524
2,Postpaid 749,946,718958.94,759.998879
3,Prepaid Entertainment 399,2142,671849.50,313.655229
4,Prepaid Data 349,1884,520066.00,276.043524


In [13]:
q("""SELECT
    circle_name,
    SUM(actual_arpu_6m) AS total_monthly_revenue
FROM customer_features
GROUP BY circle_name
ORDER BY total_monthly_revenue DESC
LIMIT 5;
""")

,circle_name,total_monthly_revenue
0,Mumbai,473288.16
1,Karnataka,468514.95
2,Andhra Pradesh,463369.80
3,North East,460998.29
4,Maharashtra,450032.30


In [14]:
q("""
SELECT tenure_years,
COUNT(*) AS subscriber_count,
AVG(actual_arpu_6m) AS avg_monthly_payment
FROM customer_features
GROUP BY tenure_years
ORDER BY tenure_years
""")

,tenure_years,subscriber_count,avg_monthly_payment
0,0,4610,329.104857
1,1,6961,342.716430
2,2,7055,350.126465
3,3,3374,357.215611


In [15]:
q("""
SELECT txn_type,
    COUNT(*) AS payments,
    COUNT(DISTINCT customer_id) AS subscribers
FROM recharges
GROUP BY txn_type
""")

,txn_type,payments,subscribers
0,Bill Payment,108419,4851
1,Recharge,299670,17149


In [16]:
q("""
SELECT AVG(amount_inr) AS average_payment_value
FROM recharges;
""")

,average_payment_value
0,532.58659


In [17]:
q("""
SELECT
    txn_type,
    ROUND(AVG(amount_inr), 2) AS average_payment,
    MIN(amount_inr) AS minimum_payment,
    MAX(amount_inr) AS maximum_payment
FROM recharges
GROUP BY txn_type
ORDER BY txn_type;
""")

,txn_type,average_payment,minimum_payment,maximum_payment
0,Bill Payment,1082.42,399,1873
1,Recharge,333.66,99,1799


In [18]:
q("""
SELECT
    connection_type,
    COUNT(*) AS subscribers,
    ROUND(AVG(recharge_count), 2) AS avg_payments_6m,
    ROUND(AVG(recharge_total_6m), 2) AS avg_recharge_total_6m
FROM customer_features
GROUP BY connection_type
ORDER BY connection_type;
""")

,connection_type,subscribers,avg_payments_6m,avg_recharge_total_6m
0,Postpaid,4851,5.39,5822.51
1,Prepaid,17149,3.36,1004.39


In [19]:
channel_counts = q("""
    SELECT
        channel,
        COUNT(*) AS payments,
        ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM recharges), 2) AS share_pct
    FROM recharges
    GROUP BY channel
    ORDER BY payments DESC
""")

channel_counts

,channel,payments,share_pct
0,Website,111163,27.24
1,Airtel App,111083,27.22
2,USSD,75257,18.44
3,Retailer,74406,18.23
4,AutoPay,36180,8.87


In [20]:
q("""
SELECT
    txn_type,
    channel,
    COUNT(*) AS no_of_payments,
    ROUND(
        100.0 * COUNT() / SUM(COUNT()) OVER (PARTITION BY txn_type),
        2
    ) AS share_pct
FROM recharges
GROUP BY txn_type, channel
ORDER BY txn_type, no_of_payments DESC
""")

,txn_type,channel,no_of_payments,share_pct
0,Bill Payment,Airtel App,36186,33.38
1,Bill Payment,AutoPay,36180,33.37
2,Bill Payment,Website,36053,33.25
3,Recharge,USSD,75257,25.11
4,Recharge,Website,75110,25.06
5,Recharge,Airtel App,74897,24.99
6,Recharge,Retailer,74406,24.83


In [21]:
q("""
    SELECT
        recharge_count,
        COUNT(*) AS subscribers,
        ROUND(AVG(is_churned) * 100.0, 2) AS is_churned_pct
    FROM customer_features
    WHERE recharge_count <= 8
    GROUP BY recharge_count
    ORDER BY recharge_count
""")


,recharge_count,subscribers,is_churned_pct
0,0,3711,44.30
1,1,2762,40.04
2,2,2339,28.35
3,3,1026,54.09
4,4,714,76.05
5,5,3407,13.77
6,6,6732,1.52
7,7,507,5.52
8,8,32,90.63


In [22]:
q("""
    SELECT
        month,
        SUM(total_mb) AS total_mb,
        ROUND(SUM(total_mb) / 1000000.0, 3) AS total_tb,
        COUNT(DISTINCT customer_id) AS distinct_subscribers
    FROM v_usage_month
    GROUP BY month
    ORDER BY month
""")


,month,total_mb,total_tb,distinct_subscribers
0,2025-01,354388812.1,354.389,21302
1,2025-02,322251036.6,322.251,21162
2,2025-03,348661061.8,348.661,20366
3,2025-04,325337697.2,325.338,19320
4,2025-05,380734888.8,380.735,18356
5,2025-06,355411873.6,355.412,17326


In [23]:
avg_daily_data = q("""
    SELECT AVG(data_mb) AS average_daily_data_mb
    FROM usage_daily
""")

print(f"Average daily data use: {avg_daily_data.loc[0, 'average_daily_data_mb']:.2f} MB")

Average daily data use: 1572.87 MB


In [24]:
q("""SELECT 
    ROUND(avg(mb_per_observed_day), 2) AS avg_obsrv_day,
    ROUND(avg(mb_per_active_day), 2) AS avg_active_day 
FROM customer_features;
""")

,avg_obsrv_day,avg_active_day
0,524.05,1404.44


In [25]:
weekday_usage = q("""
    SELECT
        strftime('%w', usage_date) AS weekday,
        COUNT(*) AS row_count,
        ROUND(AVG(data_mb), 2) AS average_data_mb
    FROM usage_daily
    GROUP BY weekday
    ORDER BY weekday
""")
    
weekday_usage

,weekday,row_count,average_data_mb
0,0,190368,1578.06
1,1,189914,1572.94
2,2,183823,1569.55
3,3,191088,1564.48
4,4,190764,1580.00
5,5,190542,1569.51
6,6,190240,1575.45


In [26]:
q("""
    SELECT
        month,
        COUNT(DISTINCT customer_id) AS subs
    FROM v_usage_month
    GROUP BY month
    ORDER BY month;
""")

,month,subs
0,2025-01,21302
1,2025-02,21162
2,2025-03,20366
3,2025-04,19320
4,2025-05,18356
5,2025-06,17326


In [27]:
q("""
    SELECT
        strftime('%Y-%m', usage_date) AS month,
        ROUND(AVG(voice_minutes), 2) AS avg_voice_minutes,
        ROUND(AVG(sms_count), 2) AS avg_sms_count
    FROM usage_daily
    WHERE usage_date >= '2025-01-01'
      AND usage_date < '2025-07-01'
    GROUP BY month
    ORDER BY month;
""")

,month,avg_voice_minutes,avg_sms_count
0,2025-01,55.56,0.75
1,2025-02,55.84,0.75
2,2025-03,56.08,0.75
3,2025-04,56.49,0.76
4,2025-05,56.70,0.76
5,2025-06,57.13,0.77


In [28]:
q("""
    SELECT
        month,
        ROUND(AVG(total_mb) / 1024.0, 2) AS avg_gb_per_subscriber,
        ROUND(AVG(usage_days), 2) AS avg_usage_days
    FROM v_usage_month
    GROUP BY month
    ORDER BY month;
""")

,month,avg_gb_per_subscriber,avg_usage_days
0,2025-01,16.25,11.39
1,2025-02,14.87,10.31
2,2025-03,16.72,11.52
3,2025-04,16.44,11.25
4,2025-05,20.26,11.73
5,2025-06,20.03,11.46


In [29]:
q("""
    SELECT
        u.month,
        ROUND(AVG(1.0 * u.total_mb / NULLIF(u.usage_days, 0)), 2) AS avg_data_mb_per_active_day
    FROM v_usage_month AS u
    JOIN customer_features AS c
        ON u.customer_id = c.customer_id
    WHERE c.is_churned = 0
    GROUP BY u.month
    ORDER BY u.month
""")

,month,avg_data_mb_per_active_day
0,2025-01,1410.58
1,2025-02,1422.15
2,2025-03,1418.06
3,2025-04,1411.76
4,2025-05,1659.45
5,2025-06,1662.24


In [30]:
q("""
SELECT
    v.month,
    COUNT(DISTINCT v.customer_id) AS subscribers,
    ROUND(AVG(1.0 * v.total_mb / NULLIF(v.usage_days, 0)), 2) AS avg_mb_per_active_day
FROM v_usage_month AS v
JOIN customer_features AS c
    ON c.customer_id = v.customer_id
WHERE c.is_churned = 0
GROUP BY v.month
ORDER BY v.month
""")

,month,subscribers,avg_mb_per_active_day
0,2025-01,16293,1410.58
1,2025-02,16588,1422.15
2,2025-03,16640,1418.06
3,2025-04,16648,1411.76
4,2025-05,16667,1659.45
5,2025-06,16660,1662.24


In [31]:
q("""
    WITH ranked AS (
        SELECT
            NTILE(4) OVER (ORDER BY mb_per_active_day) AS quartile,
            mb_per_active_day
        FROM customer_features
        WHERE mb_per_active_day > 0
    )
    SELECT
        quartile,
        COUNT(*) AS subscribers,
        MIN(mb_per_active_day) AS min_mb_per_active_day,
        MAX(mb_per_active_day) AS max_mb_per_active_day
    FROM ranked
    GROUP BY quartile
    ORDER BY quartile
""")

,quartile,subscribers,min_mb_per_active_day,max_mb_per_active_day
0,1,5499,25.46,657.32
1,2,5499,657.35,1092.83
2,3,5499,1092.84,1571.87
3,4,5499,1572.12,9719.60


In [32]:
q("""
    WITH ranked AS (
        SELECT
            NTILE(4) OVER (ORDER BY mb_per_active_day) AS quartile,
            mb_per_active_day,
            is_churned
        FROM customer_features
        WHERE mb_per_active_day > 0
    )
    SELECT
        quartile,
        COUNT(*) AS subscribers,
        ROUND(AVG(mb_per_active_day), 2) AS avg_mb_per_active_day,
        ROUND(AVG(is_churned) * 100.0, 2) AS churn_rate_pct
    FROM ranked
    GROUP BY quartile
    ORDER BY quartile
""")



,quartile,subscribers,avg_mb_per_active_day,churn_rate_pct
0,1,5499,326.55,30.28
1,2,5499,885.44,28.11
2,3,5499,1311.51,22.51
3,4,5499,3094.26,14.09


In [33]:
q("""
    SELECT
        c.connection_type,
        c.device_type,
        ROUND(AVG(u.voice_minutes), 2) AS avg_voice_minutes_per_active_day,
        ROUND(AVG(u.data_mb), 2) AS avg_mb_per_active_day,
        COUNT(*) AS active_days
    FROM usage_daily AS u
    JOIN customer_features AS c
        ON c.customer_id = u.customer_id
    GROUP BY c.connection_type, c.device_type
    ORDER BY avg_voice_minutes_per_active_day DESC
""")

,connection_type,device_type,avg_voice_minutes_per_active_day,avg_mb_per_active_day,active_days
0,Postpaid,Feature Phone,56.71,591.87,42674
1,Postpaid,Smartphone,56.33,3724.20,259745
2,Prepaid,Smartphone,56.24,1209.17,883352
3,Prepaid,Feature Phone,56.17,184.87,140968


In [34]:
q("""
select plan_name,monthly_gb,
round(avg(total_data_mb)/1024,2) as avg_gb
from customer_features
group by plan_name,monthly_gb
order by monthly_gb;""")

,plan_name,monthly_gb,avg_gb
0,Prepaid Lite 99,15.0,17.31
1,Prepaid Smart 155,30.0,34.28
2,Prepaid Value 209,30.0,35.33
3,Postpaid 399,40.0,47.13
4,Prepaid Max 479,45.0,53.06
5,Prepaid Popular 299,45.0,53.59
6,Prepaid Annual 1799,60.0,71.20
7,Prepaid Quarterly 719,60.0,70.67
8,Postpaid 499,75.0,92.82
9,Prepaid Data 349,75.0,90.76


In [35]:
q("""
    SELECT distinct
        m_score,
        COUNT(*) AS subscribers,
        ROUND(AVG(actual_arpu_6m), 2) AS avg_monthly_spend,
        ROUND(AVG(mb_per_active_day), 2) AS avg_data_mb_per_active_day,
        ROUND(AVG(is_churned) * 100.0, 2) AS churn_rate_pct
    FROM customer_features
    GROUP BY m_score
    ORDER BY m_score
""")

,m_score,subscribers,avg_monthly_spend,avg_data_mb_per_active_day,churn_rate_pct
0,1,5500,27.79,972.57,54.82
1,2,5500,157.54,778.61,19.96
2,3,5500,279.68,1116.54,11.24
3,4,5500,912.84,2749.78,9.00


In [36]:
q("""
    WITH sms_by_subscriber AS (
        SELECT
            customer_id,
            SUM(COALESCE(sms_count, 0)) AS total_sms
        FROM usage_daily
        GROUP BY customer_id
    )
    SELECT
        (SELECT ROUND(AVG(sms_count), 2)
         FROM usage_daily) AS avg_sms_per_active_day,
        (SELECT COUNT(*)
         FROM customer_features AS c
         LEFT JOIN sms_by_subscriber AS s
             ON s.customer_id = c.customer_id
         WHERE COALESCE(s.total_sms, 0) = 0) AS subscribers_who_never_sent_sms
""")

,avg_sms_per_active_day,subscribers_who_never_sent_sms
0,0.76,658


In [37]:
q("""
    SELECT
        is_roaming,
        COUNT(*) AS day_count,
        ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM usage_daily), 2) AS day_share_pct,
        COUNT(DISTINCT customer_id) AS subscriber_count
    FROM usage_daily
    GROUP BY is_roaming
    ORDER BY is_roaming
""")

,is_roaming,day_count,day_share_pct,subscriber_count
0,0,1287143,97.02,21994
1,1,39596,2.98,16233


In [38]:
q("""
    SELECT
        is_roaming,
        ROUND(AVG(data_mb), 2) AS avg_data_mb,
        ROUND(AVG(voice_minutes), 2) AS avg_voice_minutes,
        COUNT(*) AS day_count
    FROM usage_daily
    GROUP BY is_roaming
    ORDER BY is_roaming
""")

,is_roaming,avg_data_mb,avg_voice_minutes,day_count
0,0,1572.95,56.27,1287143
1,1,1570.05,56.09,39596


In [39]:
q("""
    SELECT
        connection_type,
        COUNT(*) AS subscribers,
        ROUND(AVG(roaming_days), 2) AS avg_roaming_days,
        ROUND(AVG(CASE WHEN roaming_days > 0 THEN 1.0 ELSE 0.0 END) * 100, 2)
            AS ever_roamed_pct
    FROM customer_features
    GROUP BY connection_type
    ORDER BY connection_type
""")

,connection_type,subscribers,avg_roaming_days,ever_roamed_pct
0,Postpaid,4851,1.84,74.83
1,Prepaid,17149,1.79,73.49


In [40]:
q("""
    SELECT
        circle_name,
        SUM(roaming_days) AS total_roaming_days,
        SUM(usage_days) AS total_usage_days,
        ROUND(
            100.0 * SUM(roaming_days) / NULLIF(SUM(usage_days), 0),
            2
        ) AS roaming_day_pct
    FROM customer_features
    GROUP BY circle_name
    ORDER BY roaming_day_pct DESC
    LIMIT 5
""")

,circle_name,total_roaming_days,total_usage_days,roaming_day_pct
0,Odisha,1315,42862,3.07
1,Haryana,1861,60714,3.07
2,Delhi,1445,47078,3.07
3,Karnataka,2587,84763,3.05
4,Assam,2408,79014,3.05


In [41]:
q("""
    SELECT
        CASE
            WHEN roaming_days = 0 THEN '0: None'
            WHEN roaming_days <= 3 THEN '1: 1–3 days'
            WHEN roaming_days <= 7 THEN '2: 4–7 days'
            ELSE '3: 8+ days'
        END AS roaming_band,
        COUNT(*) AS subscribers,
  
        ROUND(AVG(is_churned) * 100.0, 2) AS churn_rate_pct
    FROM customer_features
    GROUP BY roaming_band
    ORDER BY MIN(roaming_days)
""")

,roaming_band,subscribers,churn_rate_pct
0,0: None,5767,44.62
1,1: 1–3 days,12817,19.08
2,2: 4–7 days,3303,6.27
3,3: 8+ days,113,0.88


In [46]:
q("""
    SELECT
        category,
        COUNT(*) AS ticket_count,
        ROUND(
            COUNT(*) * 100.0 / (SELECT COUNT(*) FROM complaints),
            2
        ) AS share_pct
    FROM complaints
    GROUP BY category
    ORDER BY ticket_count DESC
""")


,category,ticket_count,share_pct
0,Slow Data,8035,29.94
1,Dropped Call,6365,23.71
2,Network Coverage,4885,18.20
3,Billing,3738,13.93
4,Recharge Issue,2679,9.98
5,SIM/KYC,1139,4.24


In [43]:
q("""
SELECT COUNT(*) FROM complaints
""")

,COUNT(*)
0,26841


In [47]:
q("""
    SELECT
        channel,
        COUNT(*) AS ticket_count,
        ROUND(AVG(is_resolved) * 100.0, 2) AS resolved_share_pct
    FROM complaints
    GROUP BY channel
    ORDER BY resolved_share_pct DESC
""")

,channel,ticket_count,resolved_share_pct
0,Twitter/X,6698,70.36
1,Store,6799,70.33
2,Airtel App,6616,70.04
3,Call Center,6728,69.72


In [48]:
q("""
    SELECT
        CASE WHEN is_resolved = 0 THEN 'Open' ELSE 'Resolved' END AS status,
        COUNT(*) AS ticket_count,
        ROUND(COUNT(*) * 100.0 / (SELECT COUNT(*) FROM complaints), 2) AS share_pct
    FROM complaints
    GROUP BY is_resolved
""")

,status,ticket_count,share_pct
0,Open,8021,29.88
1,Resolved,18820,70.12


In [53]:
q("""
    SELECT
        category,
        ROUND(AVG(is_resolved) * 100.0, 2) AS resolved_share_pct,
        ROUND(AVG(CASE WHEN is_resolved = 1 THEN "{time_col}" END), 2)
            AS avg_resolution_time
    FROM complaints
    GROUP BY category
""")

,category,resolved_share_pct,avg_resolution_time
0,Billing,70.87,0.0
1,Dropped Call,69.71,0.0
2,Network Coverage,69.52,0.0
3,Recharge Issue,72.23,0.0
4,SIM/KYC,68.57,0.0
5,Slow Data,69.97,0.0


In [54]:
q("""
    SELECT
        COUNT(*) AS resolved_ticket_count,
        ROUND(AVG(resolution_days), 2) AS average_days,
        MIN(resolution_days) AS minimum_days,
        MAX(resolution_days) AS maximum_days
    FROM complaints
    WHERE is_resolved = 1
""")

,resolved_ticket_count,average_days,minimum_days,maximum_days
0,18820,7.54,1.0,14.0


In [55]:
q("""
    SELECT
        CASE
            WHEN complaint_count = 0 THEN '0 complaints'
            WHEN complaint_count = 1 THEN '1 complaint'
            ELSE '2 or more complaints'
        END AS complaint_level,
        COUNT(*) AS subscribers
    FROM customer_features
    GROUP BY complaint_level
    ORDER BY MIN(complaint_count)
""")

,complaint_level,subscribers
0,0 complaints,6631
1,1 complaint,7854
2,2 or more complaints,7515


In [56]:
q("""
    SELECT
        c.circle_name,
        COUNT(t.customer_id) AS total_tickets,
        COUNT(DISTINCT c.customer_id) AS subscribers,
        ROUND(
            1.0 * COUNT(t.customer_id) / COUNT(DISTINCT c.customer_id),
            4
        ) AS complaints_per_subscriber
    FROM customer_feature s AS c
    LEFT JOIN complaints AS t
        ON t.customer_id = c.customer_id
    GROUP BY c.circle_name
    ORDER BY complaints_per_subscriber DESC
    LIMIT 5
""")

,circle_name,total_tickets,subscribers,complaints_per_subscriber
0,Tamil Nadu,944,731,1.2914
1,Bihar,1142,887,1.2875
2,Delhi,937,735,1.2748
3,Punjab,711,558,1.2742
4,Kolkata,726,572,1.2692


In [59]:
q("""
    SELECT
        strftime('%Y', activation_date) AS join_year,
        COUNT(*) AS subscribers
    FROM customer_features
    GROUP BY join_year
    ORDER BY join_year
""")


,join_year,subscribers
0,2022,6922
1,2023,6925
2,2024,7053
3,2025,1100


In [58]:
q("""
    SELECT
        strftime('%Y', activation_date) AS join_year,
        COUNT(*) AS subscribers,
        SUM(is_churned) AS churned_subscribers,
        ROUND(100.0 * SUM(is_churned) / COUNT(*), 2) AS churn_rate_pct
    FROM customer_features
    GROUP BY join_year
    ORDER BY join_year
""")

,join_year,subscribers,churned_subscribers,churn_rate_pct
0,2022,6922,1374,19.85
1,2023,6925,1632,23.57
2,2024,7053,1901,26.95
3,2025,1100,319,29.00


In [60]:
q("""
    SELECT
        tenure_years,
        COUNT(*) AS subscribers,
        ROUND(AVG(is_churned) * 100.0, 2) AS churn_rate_pct
    FROM customer_features
    GROUP BY tenure_years
    ORDER BY tenure_years
""")

,tenure_years,subscribers,churn_rate_pct
0,0,4610,28.81
1,1,6961,24.78
2,2,7055,21.84
3,3,3374,18.73


In [64]:
q("""
    SELECT
        ROUND(AVG(days_to_churn), 1) AS avg_days_to_churn,
        MIN(days_to_churn) AS min,
        MAX(days_to_churn) AS max,
        COUNT(*) AS leaved_ones
    FROM customer_features
    WHERE is_churned = 1
""")

,avg_days_to_churn,min,max,leaved_ones
0,567.2,10,1259,5226


In [65]:
q("""
    SELECT
        tenure_years,
        COUNT(*) AS subscribers,
        ROUND(AVG(actual_arpu_6m), 2) AS avg_actual_arpu_6m,
        ROUND(AVG(recharge_total_6m), 2) AS avg_recharge_total_6m
    FROM customer_features
    GROUP BY tenure_years
    ORDER BY tenure_years
""")

,tenure_years,subscribers,avg_actual_arpu_6m,avg_recharge_total_6m
0,0,4610,329.10,1974.63
1,1,6961,342.72,2056.30
2,2,7055,350.13,2100.76
3,3,3374,357.22,2143.29


In [66]:
q("""
    SELECT
        COUNT(*) AS total_subscribers,
        SUM(is_churned) AS subscribers_lost,
        ROUND(AVG(is_churned) * 100.0, 2) AS churn_rate_pct
    FROM customer_features
""")

,total_subscribers,subscribers_lost,churn_rate_pct
0,22000,5226,23.75


In [69]:
q("""
    SELECT
        connection_type,
        COUNT(*) AS subscribers,
        SUM(is_churned) AS churned_subscribers,
        ROUND(AVG(is_churned) * 100.0, 2) AS churn_rate_pct
    FROM customer_features
    GROUP BY connection_type
    ORDER BY connection_type
""")

,connection_type,subscribers,churned_subscribers,churn_rate_pct
0,Postpaid,4851,785,16.18
1,Prepaid,17149,4441,25.90


In [70]:
q("""
    SELECT
        plan_name,
        plan_type,
        COUNT(*) AS subscribers,
        ROUND(AVG(is_churned) * 100.0, 2) AS churn_rate_pct
    FROM customer_features
    GROUP BY plan_name, plan_type
    ORDER BY churn_rate_pct DESC
    LIMIT 6
""")

,plan_name,plan_type,subscribers,churn_rate_pct
0,Prepaid Lite 99,Prepaid,996,31.12
1,Prepaid Smart 155,Prepaid,1137,30.17
2,Prepaid Entertainment 399,Prepaid,2142,26.24
3,Prepaid Quarterly 719,Prepaid,2561,25.85
4,Prepaid Value 209,Prepaid,1437,25.26
5,Prepaid Annual 1799,Prepaid,2901,25.13


In [71]:
q("""
    SELECT
        circle_name,
        region,
        COUNT(*) AS subscribers,
        SUM(is_churned) AS churned_subscribers,
        ROUND(AVG(is_churned) * 100.0, 2) AS churn_rate_pct
    FROM customer_features
    GROUP BY circle_name, region
    ORDER BY churn_rate_pct DESC, circle_name
""")

,circle_name,region,subscribers,churned_subscribers,churn_rate_pct
0,Himachal Pradesh,North,1247,417,33.44
1,Maharashtra,West,1428,470,32.91
2,Andhra Pradesh,South,1412,408,28.90
3,Mumbai,West,1395,392,28.10
4,Uttar Pradesh (East),North,554,151,27.26
5,Haryana,North,1041,265,25.46
6,Jammu & Kashmir,North,933,232,24.87
7,Gujarat,West,1210,298,24.63
8,Assam,East,1301,301,23.14
9,Uttar Pradesh (West),North,1081,248,22.94


In [72]:
q("""
    SELECT
        device_type,
        COUNT(*) AS subscribers,
        ROUND(AVG(is_churned) * 100.0, 2) AS churn_rate_pct
    FROM customer_features
    GROUP BY device_type
    ORDER BY device_type
""")

,device_type,subscribers,churn_rate_pct
0,Feature Phone,3029,23.74
1,Smartphone,18971,23.76


In [73]:
q("""
    SELECT
        acquisition_channel,
        COUNT(*) AS subscribers,
        ROUND(AVG(is_churned) * 100.0, 2) AS churn_rate_pct
    FROM customer_features
    GROUP BY acquisition_channel
    ORDER BY churn_rate_pct DESC
""")

,acquisition_channel,subscribers,churn_rate_pct
0,Call Center,1770,24.63
1,Airtel App,6513,24.20
2,Website,3081,24.12
3,Doorstep,1349,23.80
4,Retailer,9287,23.15


In [74]:
q("""
    SELECT
        age_band,
        COUNT(*) AS subscribers,
        ROUND(AVG(is_churned) * 100.0, 2) AS churn_rate_pct
    FROM customer_features
    GROUP BY age_band
    ORDER BY age_band
""")

,age_band,subscribers,churn_rate_pct
0,18-24,5268,23.65
1,25-34,7467,23.70
2,35-44,4824,23.71
3,45-54,2895,23.49
4,55+,1546,24.97


In [75]:
q("""
    SELECT
        CASE
            WHEN complaint_count = 0 THEN '1: No complaints'
            WHEN COALESCE(unresolved_count, 0) = 0 THEN '2: Complaints, none open'
            ELSE '3: At least one open complaint'
        END AS complaint_group,
        COUNT(*) AS subscribers,
        ROUND(AVG(is_churned) * 100.0, 2) AS churn_rate_pct
    FROM customer_features
    GROUP BY complaint_group
    ORDER BY
        CASE
            WHEN complaint_count = 0 THEN 1
            WHEN COALESCE(unresolved_count, 0) = 0 THEN 2
            ELSE 3
        END
""")

,complaint_group,subscribers,churn_rate_pct
0,1: No complaints,6631,20.03
1,"2: Complaints, none open",8689,23.75
2,3: At least one open complaint,6680,27.46


In [77]:
q("""
    SELECT
        unresolved_count,
        COUNT(*) AS subscribers,
        ROUND(AVG(is_churned) * 100.0, 2) AS churn_rate_pct
    FROM customer_features
    GROUP BY unresolved_count
    ORDER BY unresolved_count
""")

,unresolved_count,subscribers,churn_rate_pct
0,0,15320,22.14
1,1,5515,26.07
2,2,1018,33.10
3,3,120,37.50
4,4,25,48.00
5,5,2,100.00


In [78]:
q("""
    SELECT
        r_score,
        COUNT(*) AS subscribers,
        ROUND(AVG(is_churned) * 100.0, 2) AS churn_rate_pct
    FROM customer_features
    GROUP BY r_score
    ORDER BY r_score
""")

,r_score,subscribers,churn_rate_pct
0,1,5500,91.49
1,2,5500,3.53
2,3,5500,0.00
3,4,5500,0.00


In [80]:
q("""
    SELECT
        f_score,
        COUNT(*) AS subscribers,
        ROUND(AVG(recharge_count), 2) AS avg_recharge_count,
        ROUND(AVG(is_churned) * 100.0, 2) AS churn_rate_pct
    FROM customer_features
    GROUP BY f_score
    ORDER BY f_score
""")

,f_score,subscribers,avg_recharge_count,churn_rate_pct
0,1,5500,0.33,45.44
1,2,5500,2.51,38.67
2,3,5500,5.46,7.76
3,4,5500,6.92,3.15


In [81]:
q("""
    SELECT
        m_score AS spend_quartile,
        COUNT(*) AS subscribers,
        ROUND(AVG(actual_arpu_6m), 2) AS avg_monthly_spend,
        ROUND(AVG(is_churned) * 100.0, 2) AS churn_rate_pct
    FROM customer_features
    GROUP BY m_score
    ORDER BY m_score
""")

,spend_quartile,subscribers,avg_monthly_spend,churn_rate_pct
0,1,5500,27.79,54.82
1,2,5500,157.54,19.96
2,3,5500,279.68,11.24
3,4,5500,912.84,9.00


In [82]:
q("""
    SELECT
        region,
        COUNT(*) AS subscribers,
        ROUND(SUM(actual_arpu_6m), 2) AS monthly_revenue,
        ROUND(AVG(is_churned) * 100.0, 2) AS churn_rate_pct
    FROM customer_features
    GROUP BY region
    ORDER BY churn_rate_pct DESC
""")

,region,subscribers,monthly_revenue,churn_rate_pct
0,West,4665,1571780.02,27.85
1,North,7231,2477493.21,23.73
2,East,5459,1893199.66,21.91
3,South,4645,1635737.25,21.85
